In [1]:
import os, random, shutil
from pathlib import Path
from PIL import Image

random.seed(42)

In [2]:
def transfer_and_delete_folder(src_folder, dst_folder):
    os.makedirs(dst_folder, exist_ok=True)
    for filename in os.listdir(src_folder):
        src_path = os.path.join(src_folder, filename)
        dst_path = os.path.join(dst_folder, filename)
        if os.path.isfile(src_path):
            shutil.move(src_path, dst_path)
        elif os.path.isdir(src_path):
            shutil.move(src_path, os.path.join(dst_folder, filename))
    shutil.rmtree(src_folder)

In [3]:
def copy_random_files(src_folder, dst_folder, n):
    os.makedirs(dst_folder, exist_ok=True)
    files = [f for f in os.listdir(src_folder) if os.path.isfile(os.path.join(src_folder, f))]
    if not files:
        print("No files found in source folder.")
        return
    n = min(n, len(files))
    selected_files = random.sample(files, n)
    for file in selected_files:
        shutil.copy(os.path.join(src_folder, file), os.path.join(dst_folder, file))
    print(f"Copied {n} random files.")
    return selected_files

In [4]:
def create_black_images(src_folder, dst_folder, filenames):
    os.makedirs(dst_folder, exist_ok=True)
    for fname in filenames:
        src_path = os.path.join(src_folder, fname)
        if not os.path.isfile(src_path):
            print(f"File not found: {src_path}")
            continue
        with Image.open(src_path) as img:
            width, height = img.size
        black_img = Image.new("RGB", (width, height), (0, 0, 0))
        dst_path = os.path.join(dst_folder, os.path.splitext(fname)[0] + ".png")
        black_img.save(dst_path, "PNG")
    print(f"Created {len(filenames)} empty masks.")

In [5]:
def copy_random_images_with_masks(src_folder, dst_folder, n):
    src_images = Path(src_folder) / "images"
    src_masks = Path(src_folder) / "masks"
    dst_images = Path(dst_folder) / "images"
    dst_masks = Path(dst_folder) / "masks"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_masks.mkdir(parents=True, exist_ok=True)
    image_files = [f for f in src_images.iterdir() if f.is_file()]
    if not image_files:
        print("No images found in source/images.")
        return
    n = min(n, len(image_files))
    selected_images = random.sample(image_files, n)
    for img in selected_images:
        shutil.copy(img, dst_images / img.name)
        img_stem = img.stem
        matching_masks = [m for m in src_masks.iterdir() if m.stem == img_stem]
        for mask in matching_masks:
            shutil.copy(mask, dst_masks / mask.name)
    print(f"Copied {n} images and their matching masks.")

In [6]:
def reset_dataset_folder(main_folder):
    main_path = Path(main_folder)
    if main_path.exists():
        shutil.rmtree(main_path)
    subfolders = [
        "train/images",
        "train/masks",
        "test/images",
        "test/masks"
    ]
    for sub in subfolders:
        (main_path / sub).mkdir(parents=True, exist_ok=True)
    print(f"Reset {main_folder} with empty train/test images and masks folders.")

In [7]:
DATASET_COLONDB = "data/all-datasets/cvc-colon-db"
DATASET_KSAVIR = "data/all-datasets/ksavir-seg"
DATASET_BKAI = "data/all-datasets/bkai-igh-neopolyp"
DATASET_NEG_SUNSEG = "data/all-datasets/sun-seg-versions/sunseg_neg"  # only images
DATASET_POS_SUNSEG = "data/all-datasets/sun-seg-versions/sunseg_pos"
DATASET_ETIS = "data/all-datasets/etis-larib"  # test

DATA_V1 = "data/all-datasets/mixup/v1"
DATA_V1_TRAIN = "data/all-datasets/mixup/v1/train"
DATA_V1_TEST = "data/all-datasets/mixup/v1/test"
DATA_V2 = "data/all-datasets/mixup/v2"
DATA_V2_TRAIN = "data/all-datasets/mixup/v2/train"
DATA_V2_TEST = "data/all-datasets/mixup/v2/test"

In [8]:
info_v1 = {
    "train": {
        "positive": {
            DATASET_COLONDB: 300,
            DATASET_KSAVIR: 200,
            DATASET_BKAI: 200,
        },
        "negative": {
            DATASET_NEG_SUNSEG: 300
        }
    },
    "test": {
        DATASET_ETIS: 196,
        DATASET_POS_SUNSEG: 204
    }
}

reset_dataset_folder(DATA_V1)

for src, n in info_v1["train"]["positive"].items():
    copy_random_images_with_masks(src, DATA_V1_TRAIN, n)

for src, n in info_v1["train"]["negative"].items():
    neg_images = copy_random_files(src, DATA_V1_TRAIN+"/images", n)
    create_black_images(DATA_V1_TRAIN+"/images", DATA_V1_TRAIN+"/masks", neg_images)

for src, n in info_v1["test"].items():
    copy_random_images_with_masks(src, DATA_V1_TEST, n)

Reset data/all-datasets/mixup/v1 with empty train/test images and masks folders.
Copied 300 images and their matching masks.
Copied 200 images and their matching masks.
Copied 200 images and their matching masks.
Copied 300 random files.
Created 300 empty masks.
Copied 196 images and their matching masks.
Copied 204 images and their matching masks.


In [9]:
info_v2 = {
    "train": {
        "positive": {
            DATASET_COLONDB: 300,
            DATASET_KSAVIR: 1000,
            DATASET_BKAI: 700,
        },
        "negative": {
            DATASET_NEG_SUNSEG: 1000
        }
    },
    "test": {
        DATASET_ETIS: 196,
        DATASET_POS_SUNSEG: 204
    }
}

reset_dataset_folder(DATA_V2)

for src, n in info_v2["train"]["positive"].items():
    copy_random_images_with_masks(src, DATA_V2_TRAIN, n)

for src, n in info_v2["train"]["negative"].items():
    neg_images = copy_random_files(src, DATA_V2_TRAIN+"/images", n)
    create_black_images(DATA_V2_TRAIN+"/images", DATA_V2_TRAIN+"/masks", neg_images)

for src, n in info_v2["test"].items():
    copy_random_images_with_masks(src, DATA_V2_TEST, n)

Reset data/all-datasets/mixup/v2 with empty train/test images and masks folders.
Copied 300 images and their matching masks.
Copied 1000 images and their matching masks.
Copied 700 images and their matching masks.
Copied 1000 random files.
Created 1000 empty masks.
Copied 196 images and their matching masks.
Copied 204 images and their matching masks.
